## Ball Bouncing

My first step for implementing this type of problem was to aproach it from the pure physics engine standpoint. Given python is a OOP language, it makes sense to first define a set of classes for the various features of the problem. Using pydantic, I defined a Ball class and a Environment class. Each contains various validators and physical components of the objects they represent 


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Literal
from pydantic import BaseModel, Field, computed_field, ConfigDict

class Ball(BaseModel):
    """a 3d ball with material and state"""
    model_config = ConfigDict(arbitrary_types_allowed=True)

    radius: float
    density: float
    color: str = "blue"
    elasticity: float
    max_deformation: float = 0.1
    stiffness: float = 500.0
    dilation: float = 0.0

    start_position: list[float]
    start_velocity: list[float]
    current_position: np.ndarray = Field(default_factory=lambda: np.zeros(3, dtype=float))
    current_velocity: np.ndarray = Field(default_factory=lambda: np.zeros(3, dtype=float))
    angular_velocity: np.ndarray = Field(default_factory=lambda: np.zeros(3, dtype=float))
    acceleration: np.ndarray = Field(default_factory=lambda: np.zeros(3, dtype=float))
    force: np.ndarray = Field(default_factory=lambda: np.zeros(3, dtype=float))
    torque: np.ndarray = Field(default_factory=lambda: np.zeros(3, dtype=float))

    @computed_field
    @property
    def mass(self) -> float:
        return (4.0 / 3.0) * np.pi * (self.radius ** 3) * self.density

    @computed_field
    @property
    def moment_of_inertia(self) -> float:
        return (2.0 / 5.0) * self.mass * (self.radius ** 2)

    def initialize_state(self) -> None:
        self.current_position = np.array(self.start_position, dtype=float)
        self.current_velocity = np.array(self.start_velocity, dtype=float)
        self.angular_velocity = np.zeros(3, dtype=float)
        self.acceleration = np.zeros(3, dtype=float)
        self.force = np.zeros(3, dtype=float)
        self.torque = np.zeros(3, dtype=float)
        self.dilation = 0.0


medium_types = Literal["air", "helium", "nitrogen"]
class BounceEnvironment(BaseModel):
    """environment parameters for the room"""
    model_config = ConfigDict(arbitrary_types_allowed=True)

    room_dimensions: list[float]
    gravity: float
    gravity_vector: list[float] = [0.0, 0.0, -1.0]
    linear_drag: float = 0.0
    fluid_density: float = 1.225
    drag_coefficient: float = 0.47
    medium_type: medium_types = "air"
    wall_restitution: float = 0.9
    wall_curvature: float = 0.0


Next, I defined a "Simulation" class - this is where the interactions between the an array of the Ball class and the environment are included. We include class methods for simulating the ball interactions, the ball to wall interactions, and the general environment acting on the balls (e.g., the media the balls travel through, the forces the balls experience, etc.)

There are a few limitations to this:
1. Doing this type of stepwise approach can be collisions inconsistent and wrong in behavior. Since the steps are integers, it could be that at the moment of the next "step" we have already had two balls collide, and the analysis then shows the balls at an intersected point, and redirects them with that in the state memory. That is incorrect, and could lead to incorrect "bouncing" behaviors. 
2. I used a analytical solution for the velocity and acceleration here, which could "blow up" and lead to incorrect behaviors like balls rapidly gaining momentum due to the modeled in energy leak. There are some alternatives to this that are symplectic, but I stuck for now with the pure analytical solution. 
3. Modeling things like elasticity with different ball types - currently, the elasticity calculation assumes that we use one type of material, but technically, with two materials, we would see different output elasticities (e.g., a rubber ball and a wood ball hitting each other)


Some key components that I did want to be able to physically repesent:
1. Forces on ball - I wanted to be able to model standard forces on the ball, such as gravity, and ball to ball force causing to acceleration. This resulted in having to implement specific mass balance equiations as well, given that we enabled differnet sized/mass balls in the same simulation.
2. Torque on ball - I wanted to be able to include things like Magnus forces, which required precalculating the angular velocity of the ball
3. Environmental physics - I wanted to be able to accurately assess the impacts of restitution, and how different material properties of the ball and the wall may impact the collisions

In [3]:
class SimulationSimple(BaseModel):
    """run a physics simulation for multiple balls"""
    balls: list[Ball]
    environment: BounceEnvironment
    time_step: float
    total_time_steps: int
    positions: list[np.ndarray] = Field(default_factory=list)
    velocities: list[np.ndarray] = Field(default_factory=list)
    occupancy_grid: np.ndarray | None = None
    occupancy_history: list[np.ndarray] = Field(default_factory=list)

    class Config:
        arbitrary_types_allowed = True

    def _gravity_vector(self) -> np.ndarray:
        g_dir = np.array(self.environment.gravity_vector, dtype=float)
        g_norm = np.linalg.norm(g_dir)
        if g_norm == 0.0:
            return np.zeros(3, dtype=float)
        return (g_dir / g_norm) * self.environment.gravity

    def calculate_forces(self) -> None:
        g_vec = self._gravity_vector()
        for ball in self.balls:
            ball.force = ball.mass * g_vec

            v = ball.current_velocity
            speed = np.linalg.norm(v)
            if self.environment.linear_drag != 0.0:
                ball.force += -self.environment.linear_drag * v
            if self.environment.drag_coefficient != 0.0 and speed > 0.0:
                ball.force += -self.environment.drag_coefficient * speed * v

    def update_acceleration(self) -> None:
        for ball in self.balls:
            ball.acceleration = ball.force / ball.mass

    def update_velocity(self, dt: float) -> None:
        for ball in self.balls:
            ball.current_velocity = (
                ball.current_velocity + ball.acceleration * dt
            )

    def update_position(self, dt: float) -> None:
        for ball in self.balls:
            ball.current_position = (
                ball.current_position + ball.current_velocity * dt
            )

    def update_deformation(self, dt: float) -> None:
        for ball in self.balls:
            speed = np.linalg.norm(ball.current_velocity)
            ball.dilation = ball.dilation + speed * dt

    def _compute_substeps(self) -> int:
        if len(self.balls) == 0:
            return 1

        min_radius = min(ball.radius for ball in self.balls)
        if min_radius <= 0.0:
            return 1

        max_speed = max(np.linalg.norm(ball.current_velocity) for ball in self.balls)
        target_disp = 0.25 * min_radius
        if target_disp <= 0.0:
            return 1

        substeps = int(np.ceil((max_speed * self.time_step) / target_disp))
        return max(1, min(substeps, 20))

    def bounce_off_walls(self, ball: Ball) -> None:
        dims = np.array(self.environment.room_dimensions, dtype=float)
        pos = ball.current_position
        vel = ball.current_velocity
        r = ball.radius
        restitution = self.environment.wall_restitution

        for axis in range(3):
            min_bound = r
            max_bound = dims[axis] - r
            if pos[axis] < min_bound:
                pos[axis] = min_bound
                vel[axis] = -vel[axis] * restitution
            elif pos[axis] > max_bound:
                pos[axis] = max_bound
                vel[axis] = -vel[axis] * restitution

        ball.current_position = pos
        ball.current_velocity = vel

    def handle_ball_collisions(self) -> None:
        count = len(self.balls)

        for i in range(count):
            for j in range(i + 1, count):
                b1 = self.balls[i]
                b2 = self.balls[j]
                delta = b2.current_position - b1.current_position
                dist = np.linalg.norm(delta)
                min_dist = b1.radius + b2.radius

                if dist >= min_dist:
                    continue

                if dist == 0.0:
                    normal = np.array([1.0, 0.0, 0.0], dtype=float)
                else:
                    normal = delta / dist

                overlap = min_dist - dist
                inv_m1 = 0.0 if b1.mass == 0.0 else 1.0 / b1.mass
                inv_m2 = 0.0 if b2.mass == 0.0 else 1.0 / b2.mass
                inv_mass_sum = inv_m1 + inv_m2
                if inv_mass_sum == 0.0:
                    continue

                b1.current_position -= normal * (overlap * (inv_m1 / inv_mass_sum))
                b2.current_position += normal * (overlap * (inv_m2 / inv_mass_sum))

                rel_vel = b2.current_velocity - b1.current_velocity
                vel_along_normal = np.dot(rel_vel, normal)
                if vel_along_normal > 0.0:
                    continue

                restitution = np.sqrt(max(0.0, b1.elasticity * b2.elasticity))
                impulse_mag = (-(1.0 + restitution) * vel_along_normal) / inv_mass_sum
                impulse = impulse_mag * normal

                b1.current_velocity -= impulse * inv_m1
                b2.current_velocity += impulse * inv_m2

    def _snapshot(self) -> tuple[np.ndarray, np.ndarray]:
        positions = np.stack([b.current_position for b in self.balls], axis=0)
        velocities = np.stack([b.current_velocity for b in self.balls], axis=0)
        return positions, velocities

    def init_grid(self, resolution: int = 20) -> None:
        self.occupancy_grid = np.zeros((resolution, resolution, resolution), dtype=float)

    def update_occupancy(self) -> None:
        if self.occupancy_grid is None:
            return
        dims = np.array(self.environment.room_dimensions, dtype=float)
        res = self.occupancy_grid.shape[0]
        for ball in self.balls:
            idx = ((ball.current_position / dims) * (res - 1)).astype(int)
            idx = np.clip(idx, 0, res - 1)
            self.occupancy_grid[tuple(idx)] += 1.0

    def step(self) -> None:
        substeps = self._compute_substeps()
        dt = self.time_step / substeps

        for _ in range(substeps):
            self.calculate_forces()
            self.update_acceleration()
            self.update_velocity(dt)
            self.update_position(dt)
            self.update_deformation(dt)
            for ball in self.balls:
                self.bounce_off_walls(ball)

            # run a couple collision passes to resolve chained overlaps
            for _ in range(2):
                self.handle_ball_collisions()

        positions, velocities = self._snapshot()
        self.positions.append(positions)
        self.velocities.append(velocities)
        self.update_occupancy()
        if self.occupancy_grid is not None:
            self.occupancy_history.append(self.occupancy_grid.sum(axis=2).copy())

    def simulate(self) -> None:
        for ball in self.balls:
            ball.initialize_state()
        positions, velocities = self._snapshot()
        self.positions = [positions]
        self.velocities = [velocities]
        self.init_grid()
        self.occupancy_history = []
        self.update_occupancy()
        if self.occupancy_grid is not None:
            self.occupancy_history.append(self.occupancy_grid.sum(axis=2).copy())
        for _ in range(self.total_time_steps):
            self.step()

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_1167/618230312.py:1: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class SimulationSimple(BaseModel):


In [4]:
def run_baseline_simulation():
    """simple 3d baseline with one ball"""
    import plotly.graph_objects as go

    ball = Ball(
        radius=0.5,
        density=1.0,
        color="blue",
        elasticity=0.9,
        start_position=[5.0, 5.0, 8.0],
        start_velocity=[1.0, 0.8, 0.0],
    )

    env = BounceEnvironment(
        room_dimensions=[10.0, 10.0, 10.0],
        gravity=9.81,
        gravity_vector=[0.0, 0.0, -1.0],
        linear_drag=0.0,
        drag_coefficient=0.0,
        wall_restitution=0.9,
    )

    sim = SimulationSimple(
        balls=[ball],
        environment=env,
        time_step=0.02,
        total_time_steps=600,
    )
    sim.simulate()

    positions = np.array(sim.positions)[:, 0, :]
    dims = np.array(env.room_dimensions, dtype=float)

    frames = []
    for t in range(len(positions)):
        pos = positions[t]
        frames.append(
            go.Frame(
                data=[
                    go.Scatter3d(
                        x=[pos[0]],
                        y=[pos[1]],
                        z=[pos[2]],
                        mode="markers",
                        marker=dict(size=8, color="blue", opacity=0.85),
                    )
                ],
                name=str(t),
            )
        )

    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=[positions[0, 0]],
                y=[positions[0, 1]],
                z=[positions[0, 2]],
                mode="markers",
                marker=dict(size=8, color="blue", opacity=0.85),
            )
        ],
        frames=frames,
    )

    fig.update_layout(
        scene=dict(
            xaxis=dict(range=[0, dims[0]], autorange=False),
            yaxis=dict(range=[0, dims[1]], autorange=False),
            zaxis=dict(range=[0, dims[2]], autorange=False),
            aspectmode="cube",
        ),
        margin=dict(l=0, r=0, b=0, t=30),
        updatemenus=[
            dict(
                type="buttons",
                buttons=[
                    dict(
                        label="play",
                        method="animate",
                        args=[None, {"frame": {"duration": 30, "redraw": True}, "fromcurrent": True}],
                    ),
                    dict(
                        label="pause",
                        method="animate",
                        args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}],
                    ),
                ],
            )
        ],
        title="baseline single-ball simulation",
    )

    return fig


baseline_fig = run_baseline_simulation()
baseline_fig

In [5]:
import ipywidgets as widgets
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import ast


def _random_non_overlapping_positions(radii, room_dims, max_tries=5000):
    rng = np.random.default_rng()
    positions = []

    for r in radii:
        mins = np.array([r, r, r], dtype=float)
        maxs = np.array(room_dims, dtype=float) - r
        placed = False

        for _ in range(max_tries):
            candidate = rng.uniform(mins, maxs)
            if all(np.linalg.norm(candidate - p) >= (r + p_r)
                   for p, p_r in zip(positions, radii[:len(positions)])):
                positions.append(candidate)
                placed = True
                break
        if not placed:
            raise ValueError(f"could not place ball with radius {r} without overlap")
    return positions


def _initial_positions(radii, room_dims, layout="random", max_tries=5000):
    """generate initial positions for different layouts while avoiding overlap"""
    rng = np.random.default_rng()
    dims = np.array(room_dims, dtype=float)

    if layout == "random":
        return _random_non_overlapping_positions(radii, room_dims, max_tries=max_tries)

    positions = []
    center = dims / 2.0

    for idx, r in enumerate(radii):
        placed = False
        for _ in range(max_tries):
            if layout == "cluster_center":
                spread = dims.min() * 0.2
                candidate = center + rng.normal(scale=spread, size=3)
            elif layout == "cluster_corner":
                base = np.array([r, r, r], dtype=float)
                spread = dims.min() * 0.2
                candidate = base + rng.normal(scale=spread, size=3)
            elif layout == "line":
                t = idx / max(1, len(radii) - 1)
                candidate = np.array([
                    r + t * (dims[0] - 2 * r),
                    dims[1] * 0.5,
                    dims[2] * 0.5,
                ], dtype=float)
            elif layout == "high_drop":
                candidate = np.array([
                    rng.uniform(r, dims[0] - r),
                    rng.uniform(r, dims[1] - r),
                    dims[2] * 0.8,
                ], dtype=float)
            else:
                candidate = rng.uniform([r, r, r], dims - r)

            if np.any(candidate < r) or np.any(candidate > dims - r):
                continue

            if all(np.linalg.norm(candidate - p) >= (r + p_r)
                   for p, p_r in zip(positions, radii[:len(positions)])):
                positions.append(candidate)
                placed = True
                break
        if not placed:
            raise ValueError(f"could not place ball with radius {r} for layout {layout}")

    return positions


def _build_balls(count, radius_range, room_dims, speed, density, layout="random", elasticity=0.5, color="blue"):
    rng = np.random.default_rng()
    r_min, r_max = radius_range
    radii = rng.uniform(r_min, r_max, count)

    positions = _initial_positions(radii, room_dims, layout=layout)

    balls = []
    for i in range(count):
        r = radii[i]
        m = density * (4.0 / 3.0) * np.pi * (r ** 3)

        direction = rng.normal(size=3)
        direction_norm = np.linalg.norm(direction)
        if direction_norm == 0.0:
            direction = np.array([1.0, 0.0, 0.0], dtype=float)
            direction_norm = 1.0
        velocity = (direction / direction_norm) * speed

        balls.append(
            Ball(
                radius=r,
                density=density,
                color=color,
                start_position=positions[i].tolist(),
                start_velocity=velocity.tolist(),
                elasticity=elasticity,
            )
        )
    return balls


def _compute_scalar_fields(positions, velocities, balls, env, mode="velocity"):
    values_per_frame = []

    if mode == "velocity":
        for v in velocities:
            values_per_frame.append(np.linalg.norm(v, axis=1))
    elif mode == "potential_energy":
        g_dir = np.array(env.gravity_direction, dtype=float)
        g_norm = np.linalg.norm(g_dir)
        if g_norm == 0.0:
            g_dir = np.array([0.0, 0.0, -1.0], dtype=float)
            g_norm = 1.0
        unit_g = g_dir / g_norm
        masses = np.array([b.mass for b in balls], dtype=float)

        for pos in positions:
            height = -np.dot(pos, unit_g)
            pe = masses * env.gravity * height
            values_per_frame.append(pe)
    else:
        for pos in positions:
            values_per_frame.append(np.zeros(pos.shape[0], dtype=float))

    all_vals = np.concatenate(values_per_frame) if values_per_frame else np.array([0.0])
    vmin = float(all_vals.min())
    vmax = float(all_vals.max())
    if vmax <= vmin:
        vmax = vmin + 1.0

    return values_per_frame, vmin, vmax


def _plot_with_density(sim, balls, env, room_dims, frame_stride=1, color_mode="velocity"):
    positions = sim.positions
    velocities = sim.velocities
    dims = np.array(room_dims, dtype=float)

    scene_config = dict(
        xaxis=dict(range=[0, dims[0]], autorange=False),
        yaxis=dict(range=[0, dims[1]], autorange=False),
        zaxis=dict(range=[0, dims[2]], autorange=False),
        aspectmode="manual",
        aspectratio=dict(x=1, y=dims[1] / dims[0], z=dims[2] / dims[0]),
    )

    radii = np.array([b.radius for b in balls], dtype=float)
    if radii.size == 0:
        marker_sizes = 6
    else:
        r_max = radii.max()
        if r_max <= 0.0:
            marker_sizes = 6
        else:
            marker_sizes = 4.0 + (radii / r_max) * 8.0

    scalar_values, vmin, vmax = _compute_scalar_fields(positions, velocities, balls, env, mode=color_mode)

    initial = positions[0]
    initial_vals = scalar_values[0]

    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=initial[:, 0],
                y=initial[:, 1],
                z=initial[:, 2],
                mode="markers",
                marker=dict(
                    size=marker_sizes,
                    color=initial_vals,
                    colorscale="Viridis",
                    cmin=vmin,
                    cmax=vmax,
                    opacity=0.8,
                    colorbar=dict(title=color_mode),
                ),
            )
        ]
    )

    frames = []
    for t in range(0, len(positions), frame_stride):
        pos = positions[t]
        vals = scalar_values[t]
        frames.append(
            go.Frame(
                data=[
                    go.Scatter3d(
                        x=pos[:, 0],
                        y=pos[:, 1],
                        z=pos[:, 2],
                        mode="markers",
                        marker=dict(
                            size=marker_sizes,
                            color=vals,
                            colorscale="Viridis",
                            cmin=vmin,
                            cmax=vmax,
                            opacity=0.8,
                        ),
                    )
                ],
                name=str(t),
            )
        )

    fig.update(frames=frames)

    fig.update_layout(
        scene=scene_config,
        uirevision="constant_view",
        margin=dict(l=0, r=0, b=0, t=30),
        updatemenus=[
            dict(
                type="buttons",
                buttons=[
                    dict(
                        label="play",
                        method="animate",
                        args=[
                            None,
                            {"frame": {"duration": 30, "redraw": True}, "fromcurrent": True},
                        ],
                    ),
                    dict(
                        label="pause",
                        method="animate",
                        args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}],
                    ),
                ],
            )
        ],
    )

    return fig


ball_count = widgets.IntSlider(value=50, min=1, max=1000, step=1, description="balls")
radius_range = widgets.FloatRangeSlider(
    value=[0.3, 0.8], min=0.1, max=2.0, step=0.1, description="radius range"
)
ball_density = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="density")
ball_speed = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, description="speed")

room_dims = widgets.Text(
    value="[10.0, 10.0, 10.0]",
    description="room [x,y,z]",
)

gravity = widgets.FloatSlider(value=9.81, min=0.0, max=20.0, step=0.1, description="gravity")
restitution = widgets.FloatSlider(value=0.9, min=0.0, max=1.0, step=0.05, description="bounciness")

material = widgets.Dropdown(
    options=["rubber", "steel", "foam"],
    value="rubber",
    description="material",
)

color_mode = widgets.Dropdown(
    options=[
        ("velocity", "velocity"),
        ("potential energy", "potential_energy"),
    ],
    value="velocity",
    description="color by",
)

init_layout = widgets.Dropdown(
    options=[
        ("random", "random"),
        ("cluster center", "cluster_center"),
        ("cluster corner", "cluster_corner"),
        ("line", "line"),
        ("high drop", "high_drop"),
    ],
    value="random",
    description="layout",
)

run_button = widgets.Button(description="generate sim")
output = widgets.Output()


def _run_simulation(_):
    output.clear_output(wait=True)
    with output:
        try:
            dims_val = ast.literal_eval(room_dims.value)
            dims = [float(v) for v in dims_val]
            if len(dims) != 3:
                raise ValueError
        except Exception:
            dims = [10.0, 10.0, 10.0]

        material_props = {
            "rubber": {"elasticity": 0.9, "color": "red"},
            "steel": {"elasticity": 0.6, "color": "gray"},
            "foam": {"elasticity": 0.8, "color": "orange"},
        }
        props = material_props[material.value]
        elasticity = props["elasticity"]
        color = props["color"]

        env = BounceEnvironment(
            room_dimensions=dims,
            gravity=gravity.value,
            gravity_vector=[0.0, 0.0, -1.0],
            linear_drag=0.0,
            drag_coefficient=0.0,
            wall_restitution=restitution.value,
        )
        balls = _build_balls(
            ball_count.value,
            radius_range.value,
            dims,
            ball_speed.value,
            ball_density.value,
            layout=init_layout.value,
            elasticity=elasticity,
            color=color,
        )
        sim = SimulationSimple(
            balls=balls,
            environment=env,
            time_step=0.02,
            total_time_steps=2000,
        )
        sim.simulate()
        frame_stride = max(1, len(sim.positions) // 300)
        fig = _plot_with_density(
            sim,
            balls,
            env,
            dims,
            frame_stride=frame_stride,
            color_mode=color_mode.value,
        )
        display(fig)


run_button.on_click(_run_simulation)

controls = widgets.VBox(
    [
        ball_count,
        radius_range,
        ball_density,
        ball_speed,
        room_dims,
        gravity,
        restitution,
        material,
        color_mode,
        init_layout,
        run_button,
    ]
)

ui = widgets.HBox([controls, output])
display(ui)

## Adding complexity to the simulation design

There are a few different ways I considered going after simulating this:
1. Scaling the number of balls: Can we simulate the ball bouncing directly here in the jupyter notebook? What's the limits of this with python? 
2. Enhancing the environment for a few balls: Can we create a more complex environment for the balls to bounce in?


I think these are very different problems to address, and they require very different computational tools to do so. The first question I tried to anser by moving away from pure python, and instead using some c++ to simulate the same physics, but with a faster underlying engine. I also tried using three.js, and having the physics operations happening directly in JS, rather than pre-computing and then storing the json output from the python, and then runnning in plotly, which is highly inefficient for these processes

Another thing to potentially implement would be to only check for the nearest neighbors when simulating collisions. Right now, the code is checking for every single ball vs. every single other ball. We don't need to be doing that every time. I would evaluate the relative time and space complexities of these two operations and pick the faster one.

### Attempting scaled solutions in three.js

I implemented an alternative version using three.js to see if I could increase the ball count.

In [12]:
from IPython.display import IFrame

#uv run python -m http.server 8000

IFrame("http://localhost:8001/index.html", width="100%", height=700)

In [11]:
IFrame("http://localhost:8001/index_complex.html", width="100%", height=700)

Next, I wanted to see if I could add some more complexity to the simulation 

In [5]:

class Simulation(BaseModel):
    """run a physics simulation for multiple balls"""
    model_config = ConfigDict(arbitrary_types_allowed=True)

    balls: list[Ball]
    environment: BounceEnvironment
    time_step: float
    total_time_steps: int
    positions: list[np.ndarray] = Field(default_factory=list)
    velocities: list[np.ndarray] = Field(default_factory=list)
    occupancy_grid: np.ndarray | None = None
    occupancy_history: list[np.ndarray] = Field(default_factory=list)

    def _gravity_vector(self) -> np.ndarray:
        g_dir = np.array(self.environment.gravity_vector, dtype=float)
        g_norm = np.linalg.norm(g_dir)
        if g_norm == 0.0:
            return np.zeros(3, dtype=float)
        return (g_dir / g_norm) * self.environment.gravity

    def calculate_forces(self) -> None:
        g_vec = self._gravity_vector()
        env = self.environment
        for ball in self.balls:
            ball.force = ball.mass * g_vec
            ball.torque = np.zeros(3, dtype=float)
            v = ball.current_velocity
            speed = np.linalg.norm(v)
            omega = ball.angular_velocity
            omega_mag = np.linalg.norm(omega)

            if env.linear_drag != 0.0:
                ball.force += -env.linear_drag * v

            if speed > 0.0 and env.fluid_density > 0.0:
                area = np.pi * (ball.radius ** 2)
                quad_mag = 0.5 * env.fluid_density * env.drag_coefficient * area * speed
                ball.force += -quad_mag * v

            if speed > 0.0 and omega_mag > 1e-10 and env.fluid_density > 0.0:
                c_magnus = 0.5
                ball.force += c_magnus * env.fluid_density * (ball.radius ** 3) * np.cross(omega, v)

            if omega_mag > 1e-10 and env.fluid_density > 0.0:
                c_ang_drag = 0.1
                ball.torque += -c_ang_drag * env.fluid_density * (ball.radius ** 5) * omega_mag * omega

    def update_acceleration(self) -> None:
        for ball in self.balls:
            ball.acceleration = ball.force / ball.mass

    def update_velocity(self) -> None:
        for ball in self.balls:
            ball.current_velocity = (
                ball.current_velocity + ball.acceleration * self.time_step
            )
            ball.angular_velocity = (
                ball.angular_velocity + (ball.torque / ball.moment_of_inertia) * self.time_step
            )

    def update_position(self) -> None:
        for ball in self.balls:
            ball.current_position = (
                ball.current_position + ball.current_velocity * self.time_step
            )

    def update_deformation(self) -> None:
        recovery_rate = 2.0
        for ball in self.balls:
            ball.dilation = max(0.0, ball.dilation - recovery_rate * self.time_step)

    def bounce_off_walls(self, ball: Ball) -> bool:
        dims = np.array(self.environment.room_dimensions, dtype=float)
        pos = ball.current_position.copy()
        vel = ball.current_velocity.copy()
        omega = ball.angular_velocity.copy()
        r = ball.radius
        e_eff = ball.elasticity * self.environment.wall_restitution
        hit = False
        wall_friction = 0.2
        friction_applied = False

        for axis in range(3):
            min_bound = r
            max_bound = dims[axis] - r
            e_axis = np.zeros(3, dtype=float)
            e_axis[axis] = 1.0

            colliding_low = pos[axis] < min_bound and vel[axis] < 0
            colliding_high = pos[axis] > max_bound and vel[axis] > 0

            if colliding_low or colliding_high:
                # save incoming velocity for dilation and friction before modifying
                incoming_speed = abs(vel[axis])
                r_vec = -r * e_axis if colliding_low else r * e_axis

                # compute friction using incoming velocity
                if not friction_applied and ball.mass > 0 and ball.moment_of_inertia > 0:
                    v_cp = vel + np.cross(omega, r_vec)
                    v_tang = v_cp - np.dot(v_cp, e_axis) * e_axis
                    v_tang_mag = np.linalg.norm(v_tang)
                    if v_tang_mag > 1e-10:
                        j_t = -wall_friction * v_tang
                        vel += j_t / ball.mass
                        omega += np.cross(r_vec, j_t) / ball.moment_of_inertia
                        friction_applied = True

                # apply normal impulse (bounce)
                pos[axis] = min_bound if colliding_low else max_bound
                vel[axis] = -vel[axis] * e_eff
                ball.dilation = min(ball.max_deformation, ball.dilation + incoming_speed * 0.01)
                hit = True

        ball.current_position = pos
        ball.current_velocity = vel
        ball.angular_velocity = omega
        return hit

    def handle_ball_collisions(self) -> bool:
        count = len(self.balls)
        had_collision = False

        for i in range(count):
            for j in range(i + 1, count):
                b1 = self.balls[i]
                b2 = self.balls[j]
                delta = b2.current_position - b1.current_position
                dist = np.linalg.norm(delta)
                min_dist = b1.radius + b2.radius

                if dist >= min_dist:
                    continue

                had_collision = True

                if dist == 0.0:
                    normal = np.array([1.0, 0.0, 0.0], dtype=float)
                else:
                    normal = delta / dist

                overlap = min_dist - dist
                inv_m1 = 1.0 / b1.mass if b1.mass > 0.0 else 0.0
                inv_m2 = 1.0 / b2.mass if b2.mass > 0.0 else 0.0
                total_inv_mass = inv_m1 + inv_m2
                if total_inv_mass > 0.0:
                    b1.current_position -= normal * (overlap * (inv_m1 / total_inv_mass))
                    b2.current_position += normal * (overlap * (inv_m2 / total_inv_mass))

                rel_vel = b2.current_velocity - b1.current_velocity
                vel_along_normal = np.dot(rel_vel, normal)
                if vel_along_normal > 0.0:
                    continue

                impact_speed = abs(vel_along_normal)
                b1.dilation = min(b1.max_deformation, b1.dilation + impact_speed * 0.01)
                b2.dilation = min(b2.max_deformation, b2.dilation + impact_speed * 0.01)

                e_eff = np.sqrt(b1.elasticity * b2.elasticity)
                impulse_mag = (-(1.0 + e_eff) * vel_along_normal) / total_inv_mass
                impulse = impulse_mag * normal

                b1.current_velocity -= impulse * inv_m1
                b2.current_velocity += impulse * inv_m2

                ball_friction = 0.3
                r1_cp = b1.radius * normal
                r2_cp = -b2.radius * normal
                v_cp = ((b1.current_velocity + np.cross(b1.angular_velocity, r1_cp))
                        - (b2.current_velocity + np.cross(b2.angular_velocity, r2_cp)))
                v_t = v_cp - np.dot(v_cp, normal) * normal
                v_t_mag = np.linalg.norm(v_t)

                if v_t_mag > 1e-10:
                    t_hat = v_t / v_t_mag
                    r1xt = np.cross(r1_cp, t_hat)
                    r2xt = np.cross(r2_cp, t_hat)
                    inv_m_eff_t = (inv_m1 + inv_m2
                                   + np.dot(r1xt, r1xt) / b1.moment_of_inertia
                                   + np.dot(r2xt, r2xt) / b2.moment_of_inertia)
                    j_t_needed = v_t_mag / inv_m_eff_t if inv_m_eff_t > 0 else 0.0
                    j_t = min(j_t_needed, ball_friction * abs(impulse_mag))
                    fric = -j_t * t_hat

                    b1.current_velocity += fric * inv_m1
                    b2.current_velocity -= fric * inv_m2
                    b1.angular_velocity += np.cross(r1_cp, fric) / b1.moment_of_inertia
                    b2.angular_velocity -= np.cross(r2_cp, fric) / b2.moment_of_inertia

        return had_collision

    def apply_rolling_friction(self) -> None:
        """friction and torque for balls resting on walls"""
        dims = np.array(self.environment.room_dimensions, dtype=float)
        wall_friction = 0.3
        contact_tol = 0.05

        for ball in self.balls:
            r = ball.radius
            pos = ball.current_position
            vel = ball.current_velocity
            omega = ball.angular_velocity

            for axis in range(3):
                in_contact = False
                n = np.zeros(3, dtype=float)

                if pos[axis] <= r + contact_tol:
                    n[axis] = 1.0
                    in_contact = True
                elif pos[axis] >= dims[axis] - r - contact_tol:
                    n[axis] = -1.0
                    in_contact = True

                if not in_contact:
                    continue
                r_vec = -r * n
                v_cp = vel + np.cross(omega, r_vec)
                v_t = v_cp - np.dot(v_cp, n) * n
                v_t_mag = np.linalg.norm(v_t)

                if v_t_mag < 1e-10:
                    continue

                f_normal = max(0.0, -np.dot(ball.force, n))
                if f_normal < 1e-10:
                    continue

                f_fric = wall_friction * f_normal
                impulse_mag = f_fric * self.time_step
                # cap so we don't reverse sliding direction
                impulse_mag = min(impulse_mag, ball.mass * v_t_mag * 0.5)

                t_hat = v_t / v_t_mag
                fric_impulse = -impulse_mag * t_hat

                ball.current_velocity += fric_impulse / ball.mass
                ball.angular_velocity += np.cross(r_vec, fric_impulse) / ball.moment_of_inertia

    def _snapshot(self) -> tuple[np.ndarray, np.ndarray]:
        positions = np.stack([b.current_position for b in self.balls], axis=0)
        velocities = np.stack([b.current_velocity for b in self.balls], axis=0)
        return positions, velocities

    def init_grid(self, resolution: int = 20) -> None:
        self.occupancy_grid = np.zeros((resolution, resolution, resolution), dtype=float)

    def update_occupancy(self) -> None:
        if self.occupancy_grid is None:
            return
        dims = np.array(self.environment.room_dimensions, dtype=float)
        res = self.occupancy_grid.shape[0]
        for ball in self.balls:
            idx = ((ball.current_position / dims) * (res - 1)).astype(int)
            idx = np.clip(idx, 0, res - 1)
            self.occupancy_grid[tuple(idx)] += 1.0

    def clamp_to_bounds(self) -> None:
        """position-only clamp after ball-ball separation"""
        dims = np.array(self.environment.room_dimensions, dtype=float)
        for ball in self.balls:
            r = ball.radius
            ball.current_position = np.clip(
                ball.current_position, r, dims - r,
            )

    def step(self, store: bool = True) -> None:
        self.calculate_forces()
        self.update_acceleration()
        self.update_velocity()
        self.update_position()
        self.update_deformation()
        for ball in self.balls:
            self.bounce_off_walls(ball)
        self.handle_ball_collisions()
        self.apply_rolling_friction()
        self.clamp_to_bounds()
        if store:
            positions, velocities = self._snapshot()
            self.positions.append(positions)
            self.velocities.append(velocities)
        self.update_occupancy()
        if store and self.occupancy_grid is not None:
            self.occupancy_history.append(self.occupancy_grid.sum(axis=2).copy())

    def simulate(self) -> None:
        for ball in self.balls:
            ball.initialize_state()
        positions, velocities = self._snapshot()
        self.positions = [positions]
        self.velocities = [velocities]
        self.init_grid()
        self.occupancy_history = []
        self.update_occupancy()
        if self.occupancy_grid is not None:
            self.occupancy_history.append(self.occupancy_grid.sum(axis=2).copy())
        for _ in range(self.total_time_steps):
            self.step()

    def simulate_until(self, stop_event, max_steps: int = 500_000, store_every: int = 10, max_frames: int = 3000) -> None:
        for ball in self.balls:
            ball.initialize_state()
        positions, velocities = self._snapshot()
        self.positions = [positions]
        self.velocities = [velocities]
        self.init_grid()
        self.occupancy_history = []
        self.update_occupancy()
        if self.occupancy_grid is not None:
            self.occupancy_history.append(self.occupancy_grid.sum(axis=2).copy())
        for i in range(max_steps):
            if stop_event.is_set():
                break
            store = (i % store_every == 0) and len(self.positions) < max_frames
            self.step(store=store)

In [ ]:

from typing import Literal


class ComplexEnvironmentSimulation(BaseModel):
    """ball simulation with custom container and static obstacles"""
    model_config = ConfigDict(arbitrary_types_allowed=True)

    balls: list[Ball]
    environment: BounceEnvironment
    time_step: float
    total_time_steps: int
    container_type: Literal["box", "cylinder"] = "box"
    blocks: list[dict] = Field(default_factory=list)
    sphere_obstacles: list[dict] = Field(default_factory=list)
    positions: list[np.ndarray] = Field(default_factory=list)
    velocities: list[np.ndarray] = Field(default_factory=list)

    def _gravity_vector(self) -> np.ndarray:
        g_dir = np.array(self.environment.gravity_vector, dtype=float)
        g_norm = np.linalg.norm(g_dir)
        if g_norm == 0.0:
            return np.zeros(3, dtype=float)
        return (g_dir / g_norm) * self.environment.gravity

    def _compute_substeps(self) -> int:
        if len(self.balls) == 0:
            return 1
        min_radius = min(ball.radius for ball in self.balls)
        max_speed = max(np.linalg.norm(ball.current_velocity) for ball in self.balls)
        if min_radius <= 0.0:
            return 1
        target_disp = 0.25 * min_radius
        n = int(np.ceil((max_speed * self.time_step) / max(target_disp, 1e-8)))
        return max(1, min(n, 24))

    def calculate_forces(self) -> None:
        g_vec = self._gravity_vector()
        for ball in self.balls:
            ball.force = ball.mass * g_vec
            v = ball.current_velocity
            speed = np.linalg.norm(v)
            if self.environment.linear_drag != 0.0:
                ball.force += -self.environment.linear_drag * v
            if self.environment.drag_coefficient != 0.0 and speed > 0.0:
                ball.force += -self.environment.drag_coefficient * speed * v

    def integrate(self, dt: float) -> None:
        for ball in self.balls:
            ball.acceleration = ball.force / ball.mass
            ball.current_velocity = ball.current_velocity + ball.acceleration * dt
            ball.current_position = ball.current_position + ball.current_velocity * dt

    def handle_container_collision(self, ball: Ball) -> None:
        pos = ball.current_position
        vel = ball.current_velocity
        dims = np.array(self.environment.room_dimensions, dtype=float)
        e = self.environment.wall_restitution
        r = ball.radius

        # z planes are shared by both box and cylinder modes
        z_min = r
        z_max = dims[2] - r
        if pos[2] < z_min:
            pos[2] = z_min
            if vel[2] < 0.0:
                vel[2] = -vel[2] * e
        elif pos[2] > z_max:
            pos[2] = z_max
            if vel[2] > 0.0:
                vel[2] = -vel[2] * e

        if self.container_type == "box":
            for axis in (0, 1):
                lo = r
                hi = dims[axis] - r
                if pos[axis] < lo:
                    pos[axis] = lo
                    if vel[axis] < 0.0:
                        vel[axis] = -vel[axis] * e
                elif pos[axis] > hi:
                    pos[axis] = hi
                    if vel[axis] > 0.0:
                        vel[axis] = -vel[axis] * e
            ball.current_position = pos
            ball.current_velocity = vel
            return

        # cylinder walls (axis aligned with z)
        center_xy = np.array([dims[0] * 0.5, dims[1] * 0.5], dtype=float)
        wall_radius = 0.5 * min(dims[0], dims[1])
        rel = pos[:2] - center_xy
        rho = np.linalg.norm(rel)
        max_rho = max(wall_radius - r, 0.0)

        if rho > max_rho:
            if rho < 1e-10:
                n_xy = np.array([1.0, 0.0], dtype=float)
            else:
                n_xy = rel / rho
            pos[:2] = center_xy + n_xy * max_rho

            v_n = np.dot(vel[:2], n_xy)
            if v_n > 0.0:
                vel[:2] = vel[:2] - (1.0 + e) * v_n * n_xy

        ball.current_position = pos
        ball.current_velocity = vel

    def handle_block_collision(self, ball: Ball, block: dict) -> None:
        pos = ball.current_position
        vel = ball.current_velocity
        e = self.environment.wall_restitution

        bmin = np.array(block["min"], dtype=float)
        bmax = np.array(block["max"], dtype=float)
        closest = np.clip(pos, bmin, bmax)
        delta = pos - closest
        dist = np.linalg.norm(delta)
        r = ball.radius

        if dist >= r:
            return

        if dist > 1e-10:
            normal = delta / dist
            overlap = r - dist
        else:
            dists = np.array([
                abs(pos[0] - bmin[0]),
                abs(bmax[0] - pos[0]),
                abs(pos[1] - bmin[1]),
                abs(bmax[1] - pos[1]),
                abs(pos[2] - bmin[2]),
                abs(bmax[2] - pos[2]),
            ], dtype=float)
            k = int(np.argmin(dists))
            normals = [
                np.array([-1.0, 0.0, 0.0], dtype=float),
                np.array([1.0, 0.0, 0.0], dtype=float),
                np.array([0.0, -1.0, 0.0], dtype=float),
                np.array([0.0, 1.0, 0.0], dtype=float),
                np.array([0.0, 0.0, -1.0], dtype=float),
                np.array([0.0, 0.0, 1.0], dtype=float),
            ]
            normal = normals[k]
            overlap = r

        pos += normal * overlap
        v_n = np.dot(vel, normal)
        if v_n < 0.0:
            vel -= (1.0 + e) * v_n * normal

        ball.current_position = pos
        ball.current_velocity = vel

    def handle_sphere_obstacle_collision(self, ball: Ball, obstacle: dict) -> None:
        pos = ball.current_position
        vel = ball.current_velocity
        e = self.environment.wall_restitution

        center = np.array(obstacle["center"], dtype=float)
        rad = float(obstacle["radius"])
        delta = pos - center
        dist = np.linalg.norm(delta)
        min_dist = ball.radius + rad

        if dist >= min_dist:
            return

        if dist < 1e-10:
            normal = np.array([1.0, 0.0, 0.0], dtype=float)
        else:
            normal = delta / dist

        pos += normal * (min_dist - dist)

        v_n = np.dot(vel, normal)
        if v_n < 0.0:
            vel -= (1.0 + e) * v_n * normal

        ball.current_position = pos
        ball.current_velocity = vel

    def handle_ball_collisions(self) -> None:
        count = len(self.balls)
        for i in range(count):
            for j in range(i + 1, count):
                b1 = self.balls[i]
                b2 = self.balls[j]

                delta = b2.current_position - b1.current_position
                dist = np.linalg.norm(delta)
                min_dist = b1.radius + b2.radius
                if dist >= min_dist:
                    continue

                if dist < 1e-10:
                    normal = np.array([1.0, 0.0, 0.0], dtype=float)
                else:
                    normal = delta / dist

                inv_m1 = 0.0 if b1.mass == 0.0 else 1.0 / b1.mass
                inv_m2 = 0.0 if b2.mass == 0.0 else 1.0 / b2.mass
                inv_mass_sum = inv_m1 + inv_m2
                if inv_mass_sum <= 0.0:
                    continue

                overlap = min_dist - dist
                b1.current_position -= normal * (overlap * inv_m1 / inv_mass_sum)
                b2.current_position += normal * (overlap * inv_m2 / inv_mass_sum)

                rel_vel = b2.current_velocity - b1.current_velocity
                v_n = np.dot(rel_vel, normal)
                if v_n > 0.0:
                    continue

                e = np.sqrt(max(0.0, b1.elasticity * b2.elasticity))
                j_imp = (-(1.0 + e) * v_n) / inv_mass_sum
                impulse = j_imp * normal
                b1.current_velocity -= impulse * inv_m1
                b2.current_velocity += impulse * inv_m2

    def _snapshot(self) -> tuple[np.ndarray, np.ndarray]:
        p = np.stack([b.current_position for b in self.balls], axis=0)
        v = np.stack([b.current_velocity for b in self.balls], axis=0)
        return p, v

    def step(self) -> None:
        n = self._compute_substeps()
        dt = self.time_step / n

        for _ in range(n):
            self.calculate_forces()
            self.integrate(dt)

            for ball in self.balls:
                self.handle_container_collision(ball)

            for ball in self.balls:
                for block in self.blocks:
                    self.handle_block_collision(ball, block)
                for obs in self.sphere_obstacles:
                    self.handle_sphere_obstacle_collision(ball, obs)

            for _ in range(2):
                self.handle_ball_collisions()

        p, v = self._snapshot()
        self.positions.append(p)
        self.velocities.append(v)

    def simulate(self) -> None:
        for ball in self.balls:
            ball.initialize_state()
        p0, v0 = self._snapshot()
        self.positions = [p0]
        self.velocities = [v0]
        for _ in range(self.total_time_steps):
            self.step()


def _complex_obstacle_preset(room_dims, preset):
    dims = np.array(room_dims, dtype=float)
    cx, cy, cz = dims * 0.5

    if preset == "none":
        return [], []

    if preset == "blocks":
        blocks = [
            {"min": [cx - 1.5, cy - 0.6, 1.5], "max": [cx + 1.5, cy + 0.6, 2.2]},
            {"min": [cx - 1.0, cy + 1.2, 3.0], "max": [cx + 1.0, cy + 2.0, 3.8]},
            {"min": [cx - 2.0, cy - 2.0, 5.2], "max": [cx - 0.8, cy - 1.2, 6.0]},
        ]
        return blocks, []

    if preset == "spheres":
        spheres = [
            {"center": [cx - 1.5, cy, cz], "radius": 0.8},
            {"center": [cx + 1.2, cy + 1.0, cz + 1.4], "radius": 0.9},
            {"center": [cx + 0.5, cy - 1.5, cz - 1.2], "radius": 0.7},
        ]
        return [], spheres

    # mixed
    blocks = [
        {"min": [cx - 2.0, cy - 0.5, 2.0], "max": [cx - 0.8, cy + 0.5, 4.0]},
        {"min": [cx + 0.8, cy - 1.8, 4.0], "max": [cx + 2.0, cy - 0.8, 6.0]},
    ]
    spheres = [
        {"center": [cx + 0.5, cy + 1.2, 3.0], "radius": 0.8},
        {"center": [cx - 0.4, cy - 1.2, 6.3], "radius": 0.6},
    ]
    return blocks, spheres


def _add_block_trace(fig, block, color="rgba(80,80,80,0.35)"):
    bmin = np.array(block["min"], dtype=float)
    bmax = np.array(block["max"], dtype=float)
    x = [bmin[0], bmax[0], bmax[0], bmin[0], bmin[0], bmax[0], bmax[0], bmin[0]]
    y = [bmin[1], bmin[1], bmax[1], bmax[1], bmin[1], bmin[1], bmax[1], bmax[1]]
    z = [bmin[2], bmin[2], bmin[2], bmin[2], bmax[2], bmax[2], bmax[2], bmax[2]]
    i = [0, 0, 0, 4, 4, 2, 1, 3, 0, 1, 6, 2]
    j = [1, 2, 3, 5, 6, 6, 5, 7, 4, 5, 7, 7]
    k = [2, 3, 1, 6, 7, 3, 6, 0, 5, 4, 2, 6]
    fig.add_trace(go.Mesh3d(x=x, y=y, z=z, i=i, j=j, k=k, color=color, opacity=0.35, showscale=False))


def _add_sphere_trace(fig, center, radius, color="rgba(90,90,150,0.30)"):
    u = np.linspace(0.0, 2.0 * np.pi, 20)
    v = np.linspace(0.0, np.pi, 12)
    x = center[0] + radius * np.outer(np.cos(u), np.sin(v))
    y = center[1] + radius * np.outer(np.sin(u), np.sin(v))
    z = center[2] + radius * np.outer(np.ones_like(u), np.cos(v))
    fig.add_trace(go.Surface(x=x, y=y, z=z, opacity=0.3, showscale=False, colorscale=[[0, color], [1, color]]))


def _add_container_guides(fig, room_dims, container_type):
    dims = np.array(room_dims, dtype=float)

    if container_type == "box":
        x0, y0, z0 = 0.0, 0.0, 0.0
        x1, y1, z1 = dims
        edges = [
            ([x0, x1], [y0, y0], [z0, z0]), ([x1, x1], [y0, y1], [z0, z0]),
            ([x1, x0], [y1, y1], [z0, z0]), ([x0, x0], [y1, y0], [z0, z0]),
            ([x0, x1], [y0, y0], [z1, z1]), ([x1, x1], [y0, y1], [z1, z1]),
            ([x1, x0], [y1, y1], [z1, z1]), ([x0, x0], [y1, y0], [z1, z1]),
            ([x0, x0], [y0, y0], [z0, z1]), ([x1, x1], [y0, y0], [z0, z1]),
            ([x1, x1], [y1, y1], [z0, z1]), ([x0, x0], [y1, y1], [z0, z1]),
        ]
        for ex, ey, ez in edges:
            fig.add_trace(go.Scatter3d(x=ex, y=ey, z=ez, mode="lines", line=dict(color="black", width=3), showlegend=False))
        return

    center_xy = np.array([dims[0] * 0.5, dims[1] * 0.5], dtype=float)
    radius = 0.5 * min(dims[0], dims[1])
    theta = np.linspace(0.0, 2.0 * np.pi, 80)
    x = center_xy[0] + radius * np.cos(theta)
    y = center_xy[1] + radius * np.sin(theta)

    fig.add_trace(go.Scatter3d(x=x, y=y, z=np.zeros_like(x), mode="lines", line=dict(color="black", width=4), showlegend=False))
    fig.add_trace(go.Scatter3d(x=x, y=y, z=np.full_like(x, dims[2]), mode="lines", line=dict(color="black", width=4), showlegend=False))

    for ang in [0.0, np.pi / 2.0, np.pi, 3.0 * np.pi / 2.0]:
        xv = center_xy[0] + radius * np.cos(ang)
        yv = center_xy[1] + radius * np.sin(ang)
        fig.add_trace(go.Scatter3d(x=[xv, xv], y=[yv, yv], z=[0.0, dims[2]], mode="lines", line=dict(color="black", width=2), showlegend=False))


def _plot_complex_scene(sim, balls, room_dims, container_type, blocks, sphere_obstacles, frame_stride=1):
    positions = sim.positions
    velocities = sim.velocities
    dims = np.array(room_dims, dtype=float)

    radii = np.array([b.radius for b in balls], dtype=float)
    if radii.size == 0:
        marker_sizes = 6.0
    else:
        rmax = max(radii.max(), 1e-8)
        marker_sizes = 4.0 + (radii / rmax) * 9.0

    initial = positions[0]
    initial_speed = np.linalg.norm(velocities[0], axis=1)

    all_speeds = np.concatenate([np.linalg.norm(v, axis=1) for v in velocities])
    cmin = float(all_speeds.min())
    cmax = float(all_speeds.max())
    if cmax <= cmin:
        cmax = cmin + 1.0

    fig = go.Figure()

    # index 0: animated balls
    fig.add_trace(
        go.Scatter3d(
            x=initial[:, 0],
            y=initial[:, 1],
            z=initial[:, 2],
            mode="markers",
            marker=dict(
                size=marker_sizes,
                color=initial_speed,
                colorscale="Viridis",
                cmin=cmin,
                cmax=cmax,
                opacity=0.85,
                colorbar=dict(title="speed"),
            ),
            showlegend=False,
        )
    )

    _add_container_guides(fig, room_dims, container_type)
    for block in blocks:
        _add_block_trace(fig, block)
    for obs in sphere_obstacles:
        _add_sphere_trace(fig, np.array(obs["center"], dtype=float), float(obs["radius"]))

    frames = []
    for t in range(0, len(positions), frame_stride):
        p = positions[t]
        s = np.linalg.norm(velocities[t], axis=1)
        frames.append(
            go.Frame(
                data=[
                    go.Scatter3d(
                        x=p[:, 0],
                        y=p[:, 1],
                        z=p[:, 2],
                        mode="markers",
                        marker=dict(size=marker_sizes, color=s, colorscale="Viridis", cmin=cmin, cmax=cmax, opacity=0.85),
                    )
                ],
                traces=[0],
                name=str(t),
            )
        )
    fig.update(frames=frames)

    fig.update_layout(
        scene=dict(
            xaxis=dict(range=[0, dims[0]], autorange=False),
            yaxis=dict(range=[0, dims[1]], autorange=False),
            zaxis=dict(range=[0, dims[2]], autorange=False),
            aspectmode="manual",
            aspectratio=dict(x=1, y=dims[1] / max(dims[0], 1e-8), z=dims[2] / max(dims[0], 1e-8)),
        ),
        uirevision="complex_env",
        margin=dict(l=0, r=0, b=0, t=30),
        updatemenus=[
            dict(
                type="buttons",
                buttons=[
                    dict(label="play", method="animate", args=[None, {"frame": {"duration": 30, "redraw": True}, "fromcurrent": True}]),
                    dict(label="pause", method="animate", args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]),
                ],
            )
        ],
        title="complex environment simulation",
    )

    return fig


# ui for complex scenes
complex_ball_count = widgets.IntSlider(value=80, min=5, max=300, step=5, description="balls")
complex_radius_range = widgets.FloatRangeSlider(value=[0.2, 0.5], min=0.1, max=1.0, step=0.05, description="radius")
complex_speed = widgets.FloatSlider(value=1.2, min=0.0, max=4.0, step=0.1, description="speed")
complex_density = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="density")
complex_dims = widgets.Text(value="[12.0, 12.0, 10.0]", description="room")
complex_container = widgets.Dropdown(options=["box", "cylinder"], value="box", description="container")
complex_obstacles = widgets.Dropdown(options=["none", "blocks", "spheres", "mixed"], value="mixed", description="obstacles")
complex_gravity = widgets.FloatSlider(value=9.81, min=0.0, max=20.0, step=0.1, description="gravity")
complex_restitution = widgets.FloatSlider(value=0.85, min=0.0, max=1.0, step=0.05, description="restitution")
complex_layout = widgets.Dropdown(
    options=[("random", "random"), ("cluster center", "cluster_center"), ("line", "line"), ("high drop", "high_drop")],
    value="random",
    description="layout",
)
complex_button = widgets.Button(description="generate complex sim")
complex_output = widgets.Output()


def _run_complex(_):
    complex_output.clear_output(wait=True)
    with complex_output:
        try:
            dims_val = ast.literal_eval(complex_dims.value)
            dims = [float(v) for v in dims_val]
            if len(dims) != 3:
                raise ValueError
        except Exception:
            dims = [12.0, 12.0, 10.0]

        blocks, spheres = _complex_obstacle_preset(dims, complex_obstacles.value)

        env = BounceEnvironment(
            room_dimensions=dims,
            gravity=complex_gravity.value,
            gravity_vector=[0.0, 0.0, -1.0],
            linear_drag=0.0,
            drag_coefficient=0.0,
            wall_restitution=complex_restitution.value,
        )

        balls = _build_balls(
            complex_ball_count.value,
            complex_radius_range.value,
            dims,
            complex_speed.value,
            complex_density.value,
            layout=complex_layout.value,
            elasticity=0.85,
            color="blue",
        )

        sim = ComplexEnvironmentSimulation(
            balls=balls,
            environment=env,
            time_step=0.02,
            total_time_steps=1800,
            container_type=complex_container.value,
            blocks=blocks,
            sphere_obstacles=spheres,
        )
        sim.simulate()

        frame_stride = max(1, len(sim.positions) // 320)
        fig = _plot_complex_scene(
            sim,
            balls,
            dims,
            complex_container.value,
            blocks,
            spheres,
            frame_stride=frame_stride,
        )
        display(fig)


complex_button.on_click(_run_complex)

complex_controls = widgets.VBox(
    [
        complex_ball_count,
        complex_radius_range,
        complex_speed,
        complex_density,
        complex_dims,
        complex_container,
        complex_obstacles,
        complex_layout,
        complex_gravity,
        complex_restitution,
        complex_button,
    ]
)

complex_ui = widgets.HBox([complex_controls, complex_output])
display(complex_ui)
